# 📈 Phân Tích Cổ Phiếu Tesla (TSLA)
## Lấy dữ liệu từ Yahoo Finance, EDA, Dự đoán giá & Clustering độ biến động

**Nội dung:**
1. Cài đặt và Import thư viện
2. Lấy dữ liệu cổ phiếu Tesla từ Yahoo Finance
3. Khám phá dữ liệu (EDA)
4. Dự đoán giá cổ phiếu bằng hồi quy Time Series
5. Clustering độ biến động theo thời gian

## 1. Cài đặt và Import thư viện

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install yfinance pandas numpy matplotlib seaborn scikit-learn statsmodels plotly -q

In [ ]:
# Import các thư viện
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Đã import thành công tất cả các thư viện!")

## 2. Lấy dữ liệu cổ phiếu Tesla từ Yahoo Finance

In [ ]:
# Lấy dữ liệu cổ phiếu Tesla từ Yahoo Finance
ticker = "TSLA"
start_date = "2020-01-01"
end_date = datetime.now().strftime("%Y-%m-%d")

# Download dữ liệu
print(f"📥 Đang tải dữ liệu cổ phiếu {ticker} từ {start_date} đến {end_date}...")
tesla_data = yf.download(ticker, start=start_date, end=end_date, progress=False)

# Xử lý multi-level columns nếu có
if isinstance(tesla_data.columns, pd.MultiIndex):
    tesla_data.columns = tesla_data.columns.get_level_values(0)

# Reset index để có cột Date
tesla_data.reset_index(inplace=True)

print(f"✅ Đã tải thành công {len(tesla_data)} dòng dữ liệu!")
print(f"📅 Khoảng thời gian: {tesla_data['Date'].min().strftime('%Y-%m-%d')} đến {tesla_data['Date'].max().strftime('%Y-%m-%d')}")

In [ ]:
# Xem cấu trúc dữ liệu
print("📊 Cấu trúc dữ liệu:")
print("-" * 50)
print(tesla_data.info())
print("\n📋 5 dòng đầu tiên:")
tesla_data.head()

In [ ]:
# Thống kê mô tả
print("📈 Thống kê mô tả:")
tesla_data.describe()

## 3. Khám phá dữ liệu (EDA - Exploratory Data Analysis)

In [ ]:
# Tạo các features bổ sung cho phân tích
df = tesla_data.copy()

# Tính toán các chỉ số kỹ thuật
df['Daily_Return'] = df['Close'].pct_change() * 100  # Lợi nhuận hàng ngày (%)
df['Volatility_20d'] = df['Daily_Return'].rolling(window=20).std()  # Độ biến động 20 ngày
df['MA_20'] = df['Close'].rolling(window=20).mean()  # Trung bình động 20 ngày
df['MA_50'] = df['Close'].rolling(window=50).mean()  # Trung bình động 50 ngày
df['MA_200'] = df['Close'].rolling(window=200).mean()  # Trung bình động 200 ngày

# Tính Bollinger Bands
df['BB_Upper'] = df['MA_20'] + (df['Close'].rolling(window=20).std() * 2)
df['BB_Lower'] = df['MA_20'] - (df['Close'].rolling(window=20).std() * 2)

# Tính RSI (Relative Strength Index)
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

# Tính MACD
exp1 = df['Close'].ewm(span=12, adjust=False).mean()
exp2 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = exp1 - exp2
df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()

# Thêm thông tin thời gian
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Quarter'] = df['Date'].dt.quarter

print("✅ Đã tạo các features bổ sung!")
print(f"📊 Tổng số cột: {len(df.columns)}")
df.columns.tolist()

### 3.1 Biểu đồ giá cổ phiếu theo thời gian

In [ ]:
# Biểu đồ giá cổ phiếu với các đường MA
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Chart 1: Giá đóng cửa với Moving Averages
ax1 = axes[0]
ax1.plot(df['Date'], df['Close'], label='Giá đóng cửa', color='#2196F3', linewidth=1.5)
ax1.plot(df['Date'], df['MA_20'], label='MA 20', color='#FF9800', linewidth=1, alpha=0.8)
ax1.plot(df['Date'], df['MA_50'], label='MA 50', color='#4CAF50', linewidth=1, alpha=0.8)
ax1.plot(df['Date'], df['MA_200'], label='MA 200', color='#E91E63', linewidth=1, alpha=0.8)
ax1.fill_between(df['Date'], df['BB_Upper'], df['BB_Lower'], alpha=0.1, color='#2196F3', label='Bollinger Bands')
ax1.set_ylabel('Giá (USD)', fontsize=12)
ax1.set_title('📈 Giá cổ phiếu Tesla (TSLA) với Moving Averages', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Chart 2: Khối lượng giao dịch
ax2 = axes[1]
colors = ['#4CAF50' if df['Close'].iloc[i] >= df['Open'].iloc[i] else '#F44336' for i in range(len(df))]
ax2.bar(df['Date'], df['Volume'], color=colors, alpha=0.7, width=1)
ax2.set_ylabel('Khối lượng', fontsize=12)
ax2.set_xlabel('Ngày', fontsize=12)
ax2.set_title('📊 Khối lượng giao dịch', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 Phân tích lợi nhuận và độ biến động

In [ ]:
# Phân tích lợi nhuận hàng ngày
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogram lợi nhuận hàng ngày
ax1 = axes[0, 0]
df['Daily_Return'].dropna().hist(bins=50, ax=ax1, color='#3F51B5', edgecolor='white', alpha=0.7)
ax1.axvline(df['Daily_Return'].mean(), color='red', linestyle='--', linewidth=2, label=f'Trung bình: {df["Daily_Return"].mean():.2f}%')
ax1.set_xlabel('Lợi nhuận hàng ngày (%)', fontsize=11)
ax1.set_ylabel('Tần suất', fontsize=11)
ax1.set_title('📊 Phân phối lợi nhuận hàng ngày', fontsize=12, fontweight='bold')
ax1.legend()

# 2. Độ biến động theo thời gian
ax2 = axes[0, 1]
ax2.plot(df['Date'], df['Volatility_20d'], color='#E91E63', linewidth=1)
ax2.fill_between(df['Date'], df['Volatility_20d'], alpha=0.3, color='#E91E63')
ax2.set_xlabel('Ngày', fontsize=11)
ax2.set_ylabel('Độ biến động (20 ngày)', fontsize=11)
ax2.set_title('📈 Độ biến động theo thời gian', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. Box plot lợi nhuận theo năm
ax3 = axes[1, 0]
df_clean = df.dropna(subset=['Daily_Return'])
years = sorted(df_clean['Year'].unique())
data_by_year = [df_clean[df_clean['Year'] == year]['Daily_Return'].values for year in years]
bp = ax3.boxplot(data_by_year, labels=years, patch_artist=True)
colors_box = plt.cm.viridis(np.linspace(0.2, 0.8, len(years)))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
ax3.set_xlabel('Năm', fontsize=11)
ax3.set_ylabel('Lợi nhuận hàng ngày (%)', fontsize=11)
ax3.set_title('📦 Phân phối lợi nhuận theo năm', fontsize=12, fontweight='bold')
ax3.axhline(0, color='red', linestyle='--', alpha=0.5)

# 4. RSI theo thời gian
ax4 = axes[1, 1]
ax4.plot(df['Date'], df['RSI'], color='#9C27B0', linewidth=1)
ax4.axhline(70, color='red', linestyle='--', alpha=0.7, label='Quá mua (70)')
ax4.axhline(30, color='green', linestyle='--', alpha=0.7, label='Quá bán (30)')
ax4.fill_between(df['Date'], 30, 70, alpha=0.1, color='gray')
ax4.set_xlabel('Ngày', fontsize=11)
ax4.set_ylabel('RSI', fontsize=11)
ax4.set_title('📉 Chỉ số RSI', fontsize=12, fontweight='bold')
ax4.legend(loc='upper right')
ax4.set_ylim(0, 100)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.3 Phân tích tương quan và heatmap

In [ ]:
# Ma trận tương quan
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Return', 'Volatility_20d', 'RSI', 'MACD']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdYlBu_r', center=0, 
            fmt='.2f', square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('🔥 Ma trận tương quan các chỉ số', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### 3.4 Phân tích theo thời gian (Tháng, Ngày trong tuần)

In [ ]:
# Phân tích theo tháng và ngày trong tuần
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Lợi nhuận trung bình theo tháng
monthly_returns = df.groupby('Month')['Daily_Return'].mean()
colors_month = ['#4CAF50' if x >= 0 else '#F44336' for x in monthly_returns]
ax1 = axes[0]
bars1 = ax1.bar(monthly_returns.index, monthly_returns.values, color=colors_month, edgecolor='white')
ax1.set_xlabel('Tháng', fontsize=11)
ax1.set_ylabel('Lợi nhuận trung bình (%)', fontsize=11)
ax1.set_title('📅 Lợi nhuận trung bình theo tháng', fontsize=12, fontweight='bold')
ax1.axhline(0, color='black', linewidth=0.5)
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 'T11', 'T12'])
ax1.grid(axis='y', alpha=0.3)

# 2. Lợi nhuận trung bình theo ngày trong tuần
daily_returns = df.groupby('DayOfWeek')['Daily_Return'].mean()
colors_day = ['#4CAF50' if x >= 0 else '#F44336' for x in daily_returns]
ax2 = axes[1]
bars2 = ax2.bar(daily_returns.index, daily_returns.values, color=colors_day, edgecolor='white')
ax2.set_xlabel('Ngày trong tuần', fontsize=11)
ax2.set_ylabel('Lợi nhuận trung bình (%)', fontsize=11)
ax2.set_title('📆 Lợi nhuận trung bình theo ngày trong tuần', fontsize=12, fontweight='bold')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_xticks(range(5))
ax2.set_xticklabels(['Thứ 2', 'Thứ 3', 'Thứ 4', 'Thứ 5', 'Thứ 6'])
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Thống kê chi tiết
print("📊 Thống kê lợi nhuận theo tháng:")
print(monthly_returns.round(3).to_string())
print("\n📊 Thống kê lợi nhuận theo ngày trong tuần:")
print(daily_returns.round(3).to_string())

## 4. Dự đoán giá cổ phiếu bằng hồi quy Time Series

Chúng ta sẽ sử dụng các phương pháp:
- **Linear Regression** với features kỹ thuật
- **ARIMA Model** cho time series
- **Prophet-style approach** với trend và seasonality

In [ ]:
# Import các thư viện cho mô hình dự đoán
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

print("✅ Đã import các thư viện dự đoán!")

### 4.1 Chuẩn bị dữ liệu cho mô hình

In [ ]:
# Tạo features cho dự đoán
df_model = df.copy()

# Tạo target: Giá ngày tiếp theo
df_model['Target'] = df_model['Close'].shift(-1)

# Tạo lag features (giá các ngày trước)
for lag in [1, 2, 3, 5, 7, 14, 21]:
    df_model[f'Close_Lag_{lag}'] = df_model['Close'].shift(lag)
    df_model[f'Volume_Lag_{lag}'] = df_model['Volume'].shift(lag)

# Tạo thêm features từ thời gian
df_model['DayOfYear'] = df_model['Date'].dt.dayofyear
df_model['WeekOfYear'] = df_model['Date'].dt.isocalendar().week.astype(int)

# Loại bỏ các dòng có giá trị NaN
df_model = df_model.dropna()

# Chọn features cho mô hình
feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 
                'MA_20', 'MA_50', 'RSI', 'MACD', 'Signal_Line',
                'Daily_Return', 'Volatility_20d',
                'Close_Lag_1', 'Close_Lag_2', 'Close_Lag_3', 'Close_Lag_5', 'Close_Lag_7',
                'Volume_Lag_1', 'Volume_Lag_2',
                'Month', 'DayOfWeek', 'Quarter', 'DayOfYear']

X = df_model[feature_cols]
y = df_model['Target']

print(f"📊 Số lượng mẫu: {len(X)}")
print(f"📊 Số lượng features: {len(feature_cols)}")
print(f"📊 Features: {feature_cols}")

In [ ]:
# Chia dữ liệu train/test theo thời gian (không shuffle vì là time series)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Lưu dates cho visualization
dates_train = df_model['Date'][:train_size]
dates_test = df_model['Date'][train_size:]

print(f"📊 Dữ liệu training: {len(X_train)} mẫu")
print(f"📊 Dữ liệu testing: {len(X_test)} mẫu")
print(f"📅 Training period: {dates_train.iloc[0].strftime('%Y-%m-%d')} đến {dates_train.iloc[-1].strftime('%Y-%m-%d')}")
print(f"📅 Testing period: {dates_test.iloc[0].strftime('%Y-%m-%d')} đến {dates_test.iloc[-1].strftime('%Y-%m-%d')}")

### 4.2 Training các mô hình hồi quy

In [ ]:
# Định nghĩa các mô hình
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

# Training và đánh giá các mô hình
results = {}

print("🔄 Đang training các mô hình...")
print("-" * 70)

for name, model in models.items():
    # Training
    model.fit(X_train_scaled, y_train)
    
    # Prediction
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test = model.predict(X_test_scaled)
    
    # Đánh giá
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    results[name] = {
        'model': model,
        'y_pred_test': y_pred_test,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_r2': train_r2,
        'test_r2': test_r2
    }
    
    print(f"📈 {name}:")
    print(f"   Train - RMSE: ${train_rmse:.2f}, MAE: ${train_mae:.2f}, R²: {train_r2:.4f}")
    print(f"   Test  - RMSE: ${test_rmse:.2f}, MAE: ${test_mae:.2f}, R²: {test_r2:.4f}")
    print()

print("✅ Hoàn thành training!")

### 4.3 So sánh hiệu suất các mô hình

In [ ]:
# Biểu đồ so sánh các mô hình
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

model_names = list(results.keys())
test_rmse = [results[name]['test_rmse'] for name in model_names]
test_mae = [results[name]['test_mae'] for name in model_names]
test_r2 = [results[name]['test_r2'] for name in model_names]

# Colors
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(model_names)))

# RMSE comparison
ax1 = axes[0]
bars1 = ax1.barh(model_names, test_rmse, color=colors, edgecolor='white')
ax1.set_xlabel('RMSE ($)', fontsize=11)
ax1.set_title('📊 RMSE (thấp hơn = tốt hơn)', fontsize=12, fontweight='bold')
for bar, val in zip(bars1, test_rmse):
    ax1.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'${val:.2f}', 
             va='center', fontsize=10)

# MAE comparison
ax2 = axes[1]
bars2 = ax2.barh(model_names, test_mae, color=colors, edgecolor='white')
ax2.set_xlabel('MAE ($)', fontsize=11)
ax2.set_title('📊 MAE (thấp hơn = tốt hơn)', fontsize=12, fontweight='bold')
for bar, val in zip(bars2, test_mae):
    ax2.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'${val:.2f}', 
             va='center', fontsize=10)

# R² comparison
ax3 = axes[2]
bars3 = ax3.barh(model_names, test_r2, color=colors, edgecolor='white')
ax3.set_xlabel('R² Score', fontsize=11)
ax3.set_title('📊 R² (cao hơn = tốt hơn)', fontsize=12, fontweight='bold')
ax3.set_xlim(0, 1)
for bar, val in zip(bars3, test_r2):
    ax3.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}', 
             va='center', fontsize=10)

plt.tight_layout()
plt.show()

# Tìm mô hình tốt nhất
best_model_name = max(results.keys(), key=lambda x: results[x]['test_r2'])
print(f"🏆 Mô hình tốt nhất: {best_model_name} với R² = {results[best_model_name]['test_r2']:.4f}")

### 4.4 Trực quan hóa kết quả dự đoán

In [ ]:
# Biểu đồ dự đoán vs thực tế
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Chọn mô hình tốt nhất để hiển thị
best_pred = results[best_model_name]['y_pred_test']

# Chart 1: Giá thực tế vs Dự đoán
ax1 = axes[0]
ax1.plot(dates_test.values, y_test.values, label='Giá thực tế', color='#2196F3', linewidth=2)
ax1.plot(dates_test.values, best_pred, label=f'Dự đoán ({best_model_name})', color='#FF5722', linewidth=2, linestyle='--')
ax1.fill_between(dates_test.values, y_test.values, best_pred, alpha=0.2, color='#FF5722')
ax1.set_xlabel('Ngày', fontsize=11)
ax1.set_ylabel('Giá ($)', fontsize=11)
ax1.set_title(f'📈 Giá thực tế vs Dự đoán - {best_model_name}', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Chart 2: Sai số dự đoán
ax2 = axes[1]
errors = y_test.values - best_pred
colors_err = ['#4CAF50' if e >= 0 else '#F44336' for e in errors]
ax2.bar(dates_test.values, errors, color=colors_err, alpha=0.7, width=1)
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_xlabel('Ngày', fontsize=11)
ax2.set_ylabel('Sai số ($)', fontsize=11)
ax2.set_title('📉 Sai số dự đoán (Thực tế - Dự đoán)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Thống kê sai số
print(f"📊 Thống kê sai số dự đoán:")
print(f"   Sai số trung bình: ${np.mean(errors):.2f}")
print(f"   Sai số tuyệt đối trung bình: ${np.mean(np.abs(errors)):.2f}")
print(f"   Độ lệch chuẩn sai số: ${np.std(errors):.2f}")
print(f"   Sai số max: ${np.max(np.abs(errors)):.2f}")

### 4.5 Mô hình ARIMA cho Time Series

In [ ]:
# Kiểm tra tính dừng (stationarity) của chuỗi thời gian
def adf_test(series):
    result = adfuller(series.dropna())
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print(f"Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value:.4f}")
    return result[1] < 0.05

print("📊 Kiểm tra tính dừng của chuỗi giá đóng cửa:")
is_stationary = adf_test(df['Close'])
print(f"\n→ Chuỗi {'dừng' if is_stationary else 'không dừng'} (cần differencing)")

print("\n📊 Kiểm tra tính dừng sau khi differencing (d=1):")
diff_series = df['Close'].diff().dropna()
is_stationary_diff = adf_test(diff_series)
print(f"\n→ Chuỗi sau differencing {'dừng' if is_stationary_diff else 'không dừng'}")

In [ ]:
# Training mô hình ARIMA
print("🔄 Đang training mô hình ARIMA...")

# Chuẩn bị dữ liệu cho ARIMA
close_series = df.set_index('Date')['Close']
train_arima = close_series[:train_size]
test_arima = close_series[train_size:]

# Fit ARIMA model (p=5, d=1, q=0) - có thể điều chỉnh
try:
    arima_model = ARIMA(train_arima, order=(5, 1, 0))
    arima_fitted = arima_model.fit()
    
    print("\n📊 Kết quả ARIMA:")
    print(arima_fitted.summary().tables[0])
    
    # Dự đoán
    arima_predictions = arima_fitted.forecast(steps=len(test_arima))
    
    # Đánh giá
    arima_rmse = np.sqrt(mean_squared_error(test_arima, arima_predictions))
    arima_mae = mean_absolute_error(test_arima, arima_predictions)
    arima_r2 = r2_score(test_arima, arima_predictions)
    
    print(f"\n📈 Hiệu suất ARIMA trên tập test:")
    print(f"   RMSE: ${arima_rmse:.2f}")
    print(f"   MAE: ${arima_mae:.2f}")
    print(f"   R²: {arima_r2:.4f}")
    
except Exception as e:
    print(f"⚠️ Lỗi khi training ARIMA: {e}")
    arima_predictions = None

In [ ]:
# Biểu đồ dự đoán ARIMA
if arima_predictions is not None:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    ax.plot(train_arima.index, train_arima.values, label='Training Data', color='#2196F3', linewidth=1.5)
    ax.plot(test_arima.index, test_arima.values, label='Giá thực tế (Test)', color='#4CAF50', linewidth=2)
    ax.plot(test_arima.index, arima_predictions, label='Dự đoán ARIMA', color='#FF5722', linewidth=2, linestyle='--')
    
    ax.axvline(x=test_arima.index[0], color='gray', linestyle=':', linewidth=2, label='Train/Test Split')
    ax.set_xlabel('Ngày', fontsize=11)
    ax.set_ylabel('Giá ($)', fontsize=11)
    ax.set_title('📈 Dự đoán giá Tesla bằng ARIMA', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Clustering độ biến động theo thời gian

Sử dụng K-Means clustering để phân nhóm các giai đoạn có độ biến động khác nhau.

In [ ]:
# Import các thư viện clustering
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("✅ Đã import các thư viện clustering!")

### 5.1 Chuẩn bị features cho Clustering

In [ ]:
# Tạo features cho clustering độ biến động
df_cluster = df.copy()

# Tính các chỉ số biến động
df_cluster['Volatility_5d'] = df_cluster['Daily_Return'].rolling(window=5).std()
df_cluster['Volatility_10d'] = df_cluster['Daily_Return'].rolling(window=10).std()
df_cluster['Volatility_20d'] = df_cluster['Daily_Return'].rolling(window=20).std()

# Tính range (High - Low) / Close
df_cluster['Price_Range'] = (df_cluster['High'] - df_cluster['Low']) / df_cluster['Close'] * 100

# Tính volume change
df_cluster['Volume_Change'] = df_cluster['Volume'].pct_change() * 100

# Tính Average True Range (ATR)
high_low = df_cluster['High'] - df_cluster['Low']
high_close = np.abs(df_cluster['High'] - df_cluster['Close'].shift())
low_close = np.abs(df_cluster['Low'] - df_cluster['Close'].shift())
true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df_cluster['ATR_14'] = true_range.rolling(window=14).mean()

# Loại bỏ NaN
df_cluster = df_cluster.dropna()

# Chọn features cho clustering
cluster_features = ['Volatility_5d', 'Volatility_10d', 'Volatility_20d', 'Price_Range', 'ATR_14']
X_cluster = df_cluster[cluster_features]

# Chuẩn hóa
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

print(f"📊 Số lượng mẫu cho clustering: {len(X_cluster)}")
print(f"📊 Features cho clustering: {cluster_features}")

### 5.2 Tìm số cluster tối ưu (Elbow Method)

In [ ]:
# Elbow Method để tìm số cluster tối ưu
from sklearn.metrics import silhouette_score

inertias = []
silhouette_scores = []
K_range = range(2, 11)

print("🔄 Đang tính toán Elbow Method...")

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster_scaled, kmeans.labels_))

# Vẽ biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow curve
ax1 = axes[0]
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Số cluster (k)', fontsize=11)
ax1.set_ylabel('Inertia', fontsize=11)
ax1.set_title('📊 Elbow Method', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Silhouette score
ax2 = axes[1]
ax2.plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
ax2.set_xlabel('Số cluster (k)', fontsize=11)
ax2.set_ylabel('Silhouette Score', fontsize=11)
ax2.set_title('📊 Silhouette Score', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Chọn k tối ưu dựa trên Silhouette score
optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"🏆 Số cluster tối ưu (dựa trên Silhouette Score): k = {optimal_k}")

### 5.3 Thực hiện K-Means Clustering

In [ ]:
# Thực hiện K-Means clustering với k=4 (hoặc optimal_k)
n_clusters = 4  # Có thể thay bằng optimal_k

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_cluster['Cluster'] = kmeans.fit_predict(X_cluster_scaled)

# Đặt tên cho các cluster dựa trên đặc điểm
cluster_stats = df_cluster.groupby('Cluster')[cluster_features].mean()
print("📊 Đặc điểm trung bình của mỗi cluster:")
print(cluster_stats.round(3))

# Sắp xếp cluster theo độ biến động (Volatility_20d)
cluster_order = cluster_stats['Volatility_20d'].sort_values().index.tolist()
cluster_names = {cluster_order[0]: 'Biến động thấp',
                 cluster_order[1]: 'Biến động trung bình thấp',
                 cluster_order[2]: 'Biến động trung bình cao',
                 cluster_order[3]: 'Biến động cao'}

df_cluster['Cluster_Name'] = df_cluster['Cluster'].map(cluster_names)

print("\n📋 Tên các cluster:")
for k, v in cluster_names.items():
    count = (df_cluster['Cluster'] == k).sum()
    print(f"   Cluster {k}: {v} ({count} ngày - {count/len(df_cluster)*100:.1f}%)")

### 5.4 Trực quan hóa kết quả Clustering

In [ ]:
# Trực quan hóa các cluster theo thời gian
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Màu sắc cho các cluster
cluster_colors = {0: '#4CAF50', 1: '#2196F3', 2: '#FF9800', 3: '#F44336'}
color_map = [cluster_colors[c] for c in df_cluster['Cluster']]

# Chart 1: Giá cổ phiếu với cluster
ax1 = axes[0]
scatter = ax1.scatter(df_cluster['Date'], df_cluster['Close'], c=df_cluster['Cluster'], 
                      cmap='RdYlGn_r', alpha=0.6, s=20)
ax1.set_xlabel('Ngày', fontsize=11)
ax1.set_ylabel('Giá ($)', fontsize=11)
ax1.set_title('📈 Giá cổ phiếu Tesla theo cluster biến động', fontsize=14, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax1)
cbar.set_label('Cluster')
ax1.grid(True, alpha=0.3)

# Chart 2: Volatility theo thời gian với cluster
ax2 = axes[1]
for cluster in range(n_clusters):
    mask = df_cluster['Cluster'] == cluster
    ax2.scatter(df_cluster.loc[mask, 'Date'], df_cluster.loc[mask, 'Volatility_20d'], 
                c=cluster_colors[cluster], label=cluster_names[cluster], alpha=0.6, s=20)
ax2.set_xlabel('Ngày', fontsize=11)
ax2.set_ylabel('Độ biến động 20 ngày', fontsize=11)
ax2.set_title('📊 Độ biến động theo cluster', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# Chart 3: Phân bố cluster theo thời gian (stacked area)
ax3 = axes[2]
# Tạo monthly aggregation
df_cluster['YearMonth'] = df_cluster['Date'].dt.to_period('M')
cluster_monthly = df_cluster.groupby(['YearMonth', 'Cluster']).size().unstack(fill_value=0)
cluster_monthly_pct = cluster_monthly.div(cluster_monthly.sum(axis=1), axis=0) * 100

# Convert Period to datetime for plotting
dates_plot = cluster_monthly_pct.index.to_timestamp()
cluster_monthly_pct.index = dates_plot

cluster_monthly_pct.plot(kind='area', stacked=True, ax=ax3, 
                         color=[cluster_colors[i] for i in range(n_clusters)],
                         alpha=0.7)
ax3.set_xlabel('Thời gian', fontsize=11)
ax3.set_ylabel('Phần trăm (%)', fontsize=11)
ax3.set_title('📅 Phân bố cluster theo thời gian', fontsize=14, fontweight='bold')
ax3.legend([cluster_names[i] for i in range(n_clusters)], loc='upper right')
ax3.set_ylim(0, 100)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ chi tiết về đặc điểm các cluster
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Box plot Volatility theo cluster
ax1 = axes[0, 0]
df_cluster.boxplot(column='Volatility_20d', by='Cluster_Name', ax=ax1, 
                   patch_artist=True)
ax1.set_xlabel('Cluster', fontsize=11)
ax1.set_ylabel('Độ biến động 20 ngày', fontsize=11)
ax1.set_title('📦 Phân phối độ biến động theo cluster', fontsize=12, fontweight='bold')
plt.suptitle('')

# 2. Box plot Daily Return theo cluster
ax2 = axes[0, 1]
df_cluster.boxplot(column='Daily_Return', by='Cluster_Name', ax=ax2,
                   patch_artist=True)
ax2.set_xlabel('Cluster', fontsize=11)
ax2.set_ylabel('Lợi nhuận hàng ngày (%)', fontsize=11)
ax2.set_title('📦 Phân phối lợi nhuận theo cluster', fontsize=12, fontweight='bold')
plt.suptitle('')

# 3. Scatter plot Volatility vs Price Range
ax3 = axes[1, 0]
for cluster in range(n_clusters):
    mask = df_cluster['Cluster'] == cluster
    ax3.scatter(df_cluster.loc[mask, 'Volatility_20d'], 
                df_cluster.loc[mask, 'Price_Range'],
                c=cluster_colors[cluster], label=cluster_names[cluster], alpha=0.5, s=30)
ax3.set_xlabel('Độ biến động 20 ngày', fontsize=11)
ax3.set_ylabel('Price Range (%)', fontsize=11)
ax3.set_title('📊 Volatility vs Price Range', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Radar chart cho các cluster
ax4 = axes[1, 1]
cluster_means = df_cluster.groupby('Cluster')[cluster_features].mean()
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

angles = np.linspace(0, 2*np.pi, len(cluster_features), endpoint=False).tolist()
angles += angles[:1]

for cluster in range(n_clusters):
    values = cluster_means_norm.loc[cluster].tolist()
    values += values[:1]
    ax4.plot(angles, values, 'o-', linewidth=2, label=cluster_names[cluster], 
             color=cluster_colors[cluster])
    ax4.fill(angles, values, alpha=0.1, color=cluster_colors[cluster])

ax4.set_xticks(angles[:-1])
ax4.set_xticklabels(cluster_features, size=9)
ax4.set_title('🎯 Radar Chart - Đặc điểm các cluster', fontsize=12, fontweight='bold')
ax4.legend(loc='upper right', bbox_to_anchor=(1.3, 1))

plt.tight_layout()
plt.show()

### 5.5 Phân tích chi tiết theo Cluster

In [ ]:
# Phân tích chi tiết theo cluster
print("=" * 80)
print("📊 PHÂN TÍCH CHI TIẾT THEO CLUSTER")
print("=" * 80)

for cluster in range(n_clusters):
    cluster_data = df_cluster[df_cluster['Cluster'] == cluster]
    
    print(f"\n🔹 {cluster_names[cluster].upper()}")
    print("-" * 50)
    print(f"   Số ngày: {len(cluster_data)} ({len(cluster_data)/len(df_cluster)*100:.1f}%)")
    print(f"   Giá trung bình: ${cluster_data['Close'].mean():.2f}")
    print(f"   Giá cao nhất: ${cluster_data['Close'].max():.2f}")
    print(f"   Giá thấp nhất: ${cluster_data['Close'].min():.2f}")
    print(f"   Độ biến động trung bình (20d): {cluster_data['Volatility_20d'].mean():.3f}")
    print(f"   Lợi nhuận trung bình: {cluster_data['Daily_Return'].mean():.3f}%")
    print(f"   Lợi nhuận max: {cluster_data['Daily_Return'].max():.2f}%")
    print(f"   Lợi nhuận min: {cluster_data['Daily_Return'].min():.2f}%")
    
    # Thời điểm xuất hiện nhiều nhất
    month_dist = cluster_data['Month'].value_counts().head(3)
    print(f"   Top 3 tháng xuất hiện nhiều nhất: {month_dist.index.tolist()}")

print("\n" + "=" * 80)

## 6. Tổng kết và Kết luận

In [ ]:
# Tổng kết
print("=" * 80)
print("📊 TỔNG KẾT PHÂN TÍCH CỔ PHIẾU TESLA (TSLA)")
print("=" * 80)

print(f"""
📈 1. TỔNG QUAN DỮ LIỆU:
   - Khoảng thời gian: {df['Date'].min().strftime('%Y-%m-%d')} đến {df['Date'].max().strftime('%Y-%m-%d')}
   - Tổng số ngày giao dịch: {len(df)}
   - Giá đóng cửa cao nhất: ${df['Close'].max():.2f}
   - Giá đóng cửa thấp nhất: ${df['Close'].min():.2f}
   - Giá đóng cửa trung bình: ${df['Close'].mean():.2f}

📊 2. DỰ ĐOÁN GIÁ (REGRESSION):
   - Mô hình tốt nhất: {best_model_name}
   - R² Score: {results[best_model_name]['test_r2']:.4f}
   - RMSE: ${results[best_model_name]['test_rmse']:.2f}
   - MAE: ${results[best_model_name]['test_mae']:.2f}

🎯 3. CLUSTERING ĐỘ BIẾN ĐỘNG:
   - Số cluster: {n_clusters}
   - Phân loại: Biến động thấp, Trung bình thấp, Trung bình cao, Cao
   - Silhouette Score: {silhouette_score(X_cluster_scaled, kmeans.labels_):.4f}

💡 4. INSIGHTS:
   - Tesla là cổ phiếu có độ biến động cao
   - Các giai đoạn biến động cao thường đi kèm với tin tức và sự kiện quan trọng
   - Mô hình Machine Learning có thể dự đoán xu hướng giá với độ chính xác nhất định
   - Clustering giúp xác định các giai đoạn thị trường khác nhau
""")

print("=" * 80)
print("✅ Hoàn thành phân tích!")
print("=" * 80)

---

## 📌 Ghi chú

**Các kỹ thuật đã sử dụng:**
1. **EDA (Exploratory Data Analysis):** Phân tích mô tả, trực quan hóa dữ liệu, phân tích tương quan
2. **Time Series Regression:** Linear Regression, Ridge, Lasso, Random Forest, Gradient Boosting, ARIMA
3. **Clustering:** K-Means clustering với Elbow Method và Silhouette Score

**Lưu ý:**
- Kết quả dự đoán chỉ mang tính chất tham khảo, không nên dùng làm căn cứ đầu tư
- Thị trường chứng khoán chịu ảnh hưởng của nhiều yếu tố khó dự đoán
- Cần cập nhật và điều chỉnh mô hình thường xuyên

**Cải tiến có thể thực hiện:**
- Thêm các features từ sentiment analysis (tin tức, social media)
- Sử dụng LSTM/GRU cho dự đoán time series
- Kết hợp với dữ liệu vĩ mô (lãi suất, GDP, etc.)
- Áp dụng các phương pháp ensemble learning